# Comparison

Comparison of the Seapodym-LMTL model without transport (U/V fields have been set to zero) and the Seapodym-LMTL model.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import xarray as xr

from seapopym.configuration.no_transport.configuration import NoTransportConfiguration
from seapopym.configuration.no_transport.forcing_parameter import ForcingParameter, ForcingUnit
from seapopym.configuration.no_transport.functional_group_parameter import (
    FunctionalGroupParameter,
    FunctionalGroupUnit,
    FunctionalTypeParameter,
    MigratoryTypeParameter,
)
from seapopym.model.no_transport_model import NoTransportModel
from seapopym.standard.units import StandardUnitsLabels


## Loading the data

---


In [2]:
path_forcing = "/Users/adm-lehodey/Documents/Workspace/Data/phd/SEAPODYM_LMTL/2024-12-13_-_Constant_fields_no_transport/data/data*.nc"

forcing = xr.open_mfdataset(path_forcing).load()
forcing["T"].attrs["units"] = StandardUnitsLabels.temperature.units
forcing["depth"].attrs = {"axis": "Z"}
forcing["latitude"].attrs = {"axis": "Y"}
forcing["longitude"].attrs = {"axis": "X"}
forcing["time"].attrs = {"axis": "T"}
forcing

<xarray.Dataset> Size: 1GB
Dimensions:    (time: 336, depth: 2, latitude: 180, longitude: 360)
Coordinates:
  * latitude   (latitude) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * depth      (depth) int64 16B 0 1
  * longitude  (longitude) float64 3kB -179.5 -178.5 -177.5 ... 178.5 179.5
  * time       (time) datetime64[ns] 3kB 2000-01-16 2000-01-17 ... 2000-12-16
Data variables:
    U          (time, depth, latitude, longitude) float64 348MB 0.0 0.0 ... 0.0
    V          (time, depth, latitude, longitude) float64 348MB 0.0 0.0 ... 0.0
    T          (time, depth, latitude, longitude) float64 348MB 28.0 ... 28.0
    npp        (time, latitude, longitude) float64 174MB 300.0 300.0 ... 300.0
Attributes:
    comment:  Layer bounds computed from limits

In [21]:
np.unique(forcing["npp"])

array([300.])

In [3]:
path_obs = "/Users/adm-lehodey/Documents/Workspace/Data/phd/SEAPODYM_LMTL/2024-12-13_-_Constant_fields_no_transport/output/*biomass*.nc"

observation = xr.open_mfdataset(path_obs).load()
observation

<xarray.Dataset> Size: 87MB
Dimensions:    (time: 336, latitude: 180, longitude: 360)
Coordinates:
  * time       (time) datetime64[ns] 3kB 2000-01-16T12:00:00 ... 2000-12-16T1...
  * latitude   (latitude) float32 720B -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * longitude  (longitude) float32 1kB -179.5 -178.5 -177.5 ... 178.5 179.5
Data variables:
    zpk_epi    (time, latitude, longitude) float32 87MB 0.03696 ... 0.1413
Attributes: (12/19)
    Conventions:                   CF-1.7
    comment:                       sample
    diffusion_coefficient:         0
    domain:                        global
    energy_transfer:               0.1668
    institution:                   Mercator
    ...                            ...
    source_physical_variables:     Mercator GLORYS12V1
    title:                         Gobal ocean mid trophic levels biomass (zo...
    tr_max:                        10
    tr_rate:                       -0.11
    Created:                       2025-03-10
    date_field:                    20000116

In [4]:
day_layer = 0
night_layer = 0
tr_0 = 10.38
gamma_tr = -0.11
lambda_temperature_0 = 1 / 150
gamma_lambda_temperature = 0.15

f_groups = FunctionalGroupParameter(
    functional_group=[
        FunctionalGroupUnit(
            name=f"D{day_layer}N{night_layer}",
            energy_transfert=0.1668,
            migratory_type=MigratoryTypeParameter(day_layer=day_layer, night_layer=night_layer),
            functional_type=FunctionalTypeParameter(
                lambda_temperature_0=lambda_temperature_0,
                gamma_lambda_temperature=gamma_lambda_temperature,
                tr_0=tr_0,
                gamma_tr=gamma_tr,
            ),
        )
    ]
)

p_param = ForcingParameter(
    temperature=ForcingUnit(forcing=forcing["T"]),
    primary_production=ForcingUnit(forcing=forcing["npp"]),
    timestep="1D",
)

parameters = NoTransportConfiguration(forcing=p_param, functional_group=f_groups)
zooplankton_model = NoTransportModel.from_configuration(configuration=parameters)

2025-05-28 10:47:18,434 :: Seapodym ::  WARNING ::
|	npp unit is milligram / day / meter ** 2, it will be converted to kilogram / day / meter ** 2.

2025-05-28 10:47:18,471 :: Seapodym ::  WARNING ::
|	npp unit is milligram / day / meter ** 2, it will be converted to kilogram / day / meter ** 2.



In [5]:
zooplankton_model.run()

In [6]:
zooplankton_model.state

<xarray.Dataset> Size: 2GB
Dimensions:                       (time: 336, latitude: 180, longitude: 360,
                                   functional_group: 1, cohort: 11, depth: 2)
Coordinates:
  * time                          (time) datetime64[ns] 3kB 2000-01-16 ... 20...
  * latitude                      (latitude) float64 1kB -89.5 -88.5 ... 89.5
  * longitude                     (longitude) float64 3kB -179.5 ... 179.5
  * functional_group              (functional_group) int64 8B 0
  * cohort                        (cohort) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * depth                         (depth) int64 16B 0 1
Data variables: (12/28)
    biomass                       (functional_group, time, latitude, longitude) float64 174MB ...
    recruited                     (functional_group, time, latitude, longitude) float64 174MB ...
    mortality_field               (functional_group, time, latitude, longitude) float64 174MB ...
    mask_temperature              (functional_group, time, latitude, longitude, cohort) bool 240MB ...
    min_temperature               (functional_group, cohort) float64 88B 21.2...
    primary_production_by_fgroup  (functional_group, time, latitude, longitude) float64 174MB ...
    ...                            ...
    max_timestep                  (functional_group, cohort) int64 88B 1 ... 11
    mean_timestep                 (functional_group, cohort) float64 88B 1.0 ...
    timestep                      int64 8B 1
    angle_horizon_sun             float64 8B 0.0
    compute_initial_conditions    bool 1B False
    compute_preproduction         bool 1B False

## Comparison

---


### 1.a. Convert obs and pred data to comparable units


In [7]:
obs_biomass = observation["zpk_epi"].pint.quantify().pint.dequantify()
obs_biomass = obs_biomass.assign_coords({"time": obs_biomass.indexes["time"].ceil("D")})

In [8]:
pred_biomass = zooplankton_model.state.biomass
pred_biomass = pred_biomass.pint.quantify().pint.to("g m^-2").pint.dequantify().sel(functional_group=0)

### 1.b. Same for the production


In [9]:
obs_recruited = xr.open_mfdataset(
    "/Users/adm-lehodey/Documents/Workspace/Data/phd/SEAPODYM_LMTL/2024-12-13_-_Constant_fields_no_transport/output/*recprod*.nc"
).load()

In [10]:
pred_recruited = (
    zooplankton_model.state.recruited.pint.quantify().pint.to("g day-1 m^-2").pint.dequantify().sel(functional_group=0)
)

In [11]:
longitude_slice = slice(None, None)
mean_over_time = pd.DataFrame(
    {
        "Seapodym-LMTL": obs_recruited.zpk_epi_recprod.sel(longitude=longitude_slice).mean(["latitude", "longitude"]),
        "SeapoPym": pred_recruited.sel(longitude=longitude_slice).mean(["latitude", "longitude"]),
    },
    index=obs_biomass.time,
).reset_index(names="time")

In [12]:
px.line(
    mean_over_time,
    x="time",
    y=["Seapodym-LMTL", "SeapoPym"],
    title="Observed biomass",
).update_yaxes(title_text="Biomass (g m<sup>-2</sup>)").show()

### 2. Calculate the error metrics


In [13]:
longitude_slice = slice(None, None)
mean_over_time = pd.DataFrame(
    {
        "Seapodym-LMTL": obs_biomass.sel(longitude=longitude_slice).mean(["latitude", "longitude"]),
        "SeapoPym": pred_biomass.sel(longitude=longitude_slice).mean(["latitude", "longitude"]),
    },
    index=obs_biomass.time,
).reset_index(names="time")

In [14]:
px.line(
    mean_over_time,
    x="time",
    y=["Seapodym-LMTL", "SeapoPym"],
    title="Observed biomass",
).update_yaxes(title_text="Biomass (g m<sup>-2</sup>)").show()

In [15]:
diff = (
    np.abs(obs_biomass.sel(longitude=longitude_slice) - pred_biomass.sel(longitude=longitude_slice)) / obs_biomass * 100
)
time_slice = slice(None, "2000-03")
diff = diff.sel(time=time_slice)

In [16]:
mean = diff.mean(["latitude", "longitude"])
min_ = diff.min(["latitude", "longitude"])
max_ = diff.max(["latitude", "longitude"])

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=diff.time, y=mean, fill="tonexty", mode="lines", line_color="orange", name="mean (10 days windows)")
)
fig.update_layout(title="Relative difference (%)", xaxis_title="Time", yaxis_title="Relative difference (%)")
fig.update_yaxes(type="log")
fig.show()